In [ ]:
# 1. Prepare data (concatenate, add label)
real_df_copy = df.copy()
synthetic_df_copy = synthetic_df.copy()

real_df_copy['_is_real'] = 1
synthetic_df_copy['_is_real'] = 0

combined = pd.concat([real_df_copy, synthetic_df_copy], ignore_index=True)

# 2. Shuffle and split
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

y = combined['_is_real']
X = combined.drop(columns=['_is_real'])

# If you encoded/normalized data for CTGAN, use the same pipeline here.
# Ensure X contains only numeric features (or encode categoricals similarly).
# For example, if you used sklearn ColumnTransformer, apply same transform.

# Simple train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 3. Train classifier (RandomForest is robust to mixed types)
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 4. Evaluate
y_pred_proba = clf.predict_proba(X_test)[:,1]
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
prec, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')

print(f"Real-vs-Synthetic classifier - Accuracy: {acc:.3f}, AUC: {auc:.3f}, Precision: {prec:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

# 5. Plot ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1], '--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Real vs Synthetic ROC")
plt.legend()
plt.show()

# 6. Feature importance (what reveals syntheticness)
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features distinguishing real vs synthetic:\n", importances.head(20))


from scipy.stats import ks_2samp, chi2_contingency

# For numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'FraudFound_P']  # drop target if present

ks_results = {}
for col in numeric_cols:
    r = df[col].dropna()
    s = synthetic_df[col].dropna()
    stat, p = ks_2samp(r, s)
    ks_results[col] = (stat, p)

# Report worst matches (largest stat)
ks_sorted = sorted(ks_results.items(), key=lambda x: x[1][0], reverse=True)
print("Top numeric mismatches (KS statistic):")
for col, (stat, p) in ks_sorted[:10]:
    print(f"{col}: KS={stat:.3f}, p={p:.3g}")

# For categorical columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

chi2_results = {}
for col in cat_cols:
    real_counts = df[col].value_counts().reindex(df[col].unique(), fill_value=0)
    synth_counts = synthetic_df[col].value_counts().reindex(df[col].unique(), fill_value=0)
    # Build contingency table
    table = pd.concat([real_counts, synth_counts], axis=1).fillna(0).astype(int)
    table.columns = ['real','synth']
    # Chi-square requires a 2 x k table
    chi2, p, dof, exp = chi2_contingency(table.T)
    chi2_results[col] = (chi2, p)

print("Categorical columns chi2 p-values (lower means more different):")
for col, (chi2, p) in sorted(chi2_results.items(), key=lambda x: x[1][1])[:10]:
    print(f"{col}: chi2={chi2:.1f}, p={p:.3g}")
